# Análises comparativas de algoritmos

Este notebook propõe análises interessantes para comparar algoritmos no desafio SBPO 2025.

As análises usam os arquivos de resumo em `results/*/summary_*.csv` e o baseline em `best_solutions/best_objectives.csv`.

## Ideias de análise incluídas

1. **Cobertura de viabilidade e timeout** por algoritmo (confiabilidade).
2. **Gap relativo para o melhor conhecido** por instância e dataset.
3. **Fronteira eficiência x qualidade** (objetivo médio vs tempo médio).
4. **Robustez entre execuções** com variância/desvio do objetivo.
5. **Estabilidade de ranking** por instância e ranking agregado.
6. **Sensibilidade por tamanho de instância** (quando disponível em metadados).

In [ ]:
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 180)

In [ ]:
# 1) Carrega todos os summaries disponíveis
summary_paths = sorted(glob.glob('../results/*/summary_*.csv'))
if not summary_paths:
    raise FileNotFoundError('Nenhum arquivo summary_*.csv encontrado em ../results/*')

summaries = [pd.read_csv(path) for path in summary_paths]
df = pd.concat(summaries, ignore_index=True)

# 2) Carrega o melhor conhecido por instância
best = pd.read_csv('../best_solutions/best_objectives.csv')

# 3) Enriquecimento para análises
df = df.merge(best, on=['dataset', 'instance'], how='left')
df['gap_to_best_pct'] = np.where(
    df['best_objective'] > 0,
    100.0 * (df['best_objective'] - df['objective_mean']) / df['best_objective'],
    np.nan,
)
df['feasible_rate'] = np.where(df['total_runs'] > 0, df['feasible_runs'] / df['total_runs'], np.nan)
df['timeout_rate'] = np.where(df['total_runs'] > 0, df['timed_out_runs'] / df['total_runs'], np.nan)

print(f'Resumos carregados: {len(summary_paths)}')
print(f'Linhas: {len(df)} | Algoritmos: {df.algorithm.nunique()} | Instâncias: {df.instance.nunique()}')
df.head(3)

## 1) Confiabilidade: viabilidade e timeout

In [ ]:
reliability = (
    df.groupby('algorithm', as_index=False)
      .agg(
          feasible_rate_mean=('feasible_rate', 'mean'),
          timeout_rate_mean=('timeout_rate', 'mean'),
      )
      .sort_values(['feasible_rate_mean', 'timeout_rate_mean'], ascending=[False, True])
)
reliability

## 2) Qualidade: gap para melhor conhecido (quanto menor, melhor)

In [ ]:
gap_summary = (
    df.groupby(['dataset', 'algorithm'], as_index=False)
      .agg(
          avg_gap_pct=('gap_to_best_pct', 'mean'),
          median_gap_pct=('gap_to_best_pct', 'median'),
      )
      .sort_values(['dataset', 'avg_gap_pct'])
)
gap_summary

## 3) Eficiência: objetivo médio vs tempo médio (fronteira de Pareto)

In [ ]:
eff = (
    df.groupby('algorithm', as_index=False)
      .agg(
          objective_mean=('objective_mean', 'mean'),
          exec_time_mean=('exec_time_mean', 'mean'),
      )
)

x_range = max(eff['exec_time_mean'].max() - eff['exec_time_mean'].min(), 1e-9)
y_range = max(eff['objective_mean'].max() - eff['objective_mean'].min(), 1e-9)

plt.figure(figsize=(9, 6))
for idx, row in eff.iterrows():
    plt.scatter(row['exec_time_mean'], row['objective_mean'])
    dx = 0.01 * x_range * (idx % 3)
    dy = 0.01 * y_range * (idx % 2)
    plt.text(row['exec_time_mean'] + dx, row['objective_mean'] + dy, row['algorithm'], ha='left', va='bottom', fontsize=9)

plt.xlabel('Tempo médio (s)')
plt.ylabel('Objetivo médio')
plt.title('Eficiência dos algoritmos: qualidade x tempo')
plt.grid(alpha=0.3)
plt.show()

## 4) Robustez: variabilidade do objetivo

In [ ]:
robustness = (
    df.groupby('algorithm', as_index=False)
      .agg(
          objective_std_mean=('objective_std_dev', 'mean'),
          objective_var_mean=('objective_variance', 'mean'),
      )
      .sort_values('objective_std_mean')
)
robustness

## 5) Ranking composto (qualidade + confiabilidade + tempo)

In [ ]:
score = df.groupby('algorithm', as_index=False).agg(
    objective=('objective_mean', 'mean'),
    gap=('gap_to_best_pct', 'mean'),
    feasible=('feasible_rate', 'mean'),
    timeout=('timeout_rate', 'mean'),
    time=('exec_time_mean', 'mean'),
)

# Rank 1 = melhor para cada métrica
score['r_objective'] = score['objective'].rank(ascending=False, method='average')
score['r_gap'] = score['gap'].rank(ascending=True, method='average')
score['r_feasible'] = score['feasible'].rank(ascending=False, method='average')
score['r_timeout'] = score['timeout'].rank(ascending=True, method='average')
score['r_time'] = score['time'].rank(ascending=True, method='average')

# Menor score final = melhor algoritmo (média ponderada de posições)
score['final_rank_score'] = (
    0.40 * score['r_objective'] +
    0.25 * score['r_gap'] +
    0.20 * score['r_feasible'] +
    0.10 * score['r_timeout'] +
    0.05 * score['r_time']
)

final_ranking = score.sort_values('final_rank_score')
final_ranking

## Próximos passos sugeridos

- Repetir as análises separando por dataset (`a`, `b`, `x`).
- Criar gráficos por instância para detectar algoritmos especialistas.
- Adicionar testes estatísticos (Wilcoxon/Friedman + pós-teste) para comparar pares de algoritmos.
- Incluir custo computacional total (tempo x taxa de timeout) para cenários de produção.